**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2019"], check=True)
    os.chdir("/content/encore/entries/tingley-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Tingley and Buzsaki (2019) from DANDI:000218: phase locking to each neuron's own shank

Tingley D. and Buzsaki G. (2019). Routing of hippocampal ripples to subcortical structures via the lateral septum. Neuron 105, 138-149. doi:10.1016/j.neuron.2019.10.012.

Dataset: DANDI:000218, version 0.261110.1911 (doi:10.48324/dandi.000218/0.261110.1911), streamed from the archive. The version is chosen as in the main notebook's header.

Claims file: `claims/tingley-2019.yaml`, written from the accepted manuscript in PubMed Central (PMC6952543) with its supplement.

This notebook gives one reading of fig1d-phase-locking for every LS neuron: phase locking to the 120-200 Hz band of a channel on the
neuron's own shank. The LFP notebook gives this reading only for shanks inside the detection channel's 16-channel storage block
(1299 of 2327 neurons), as the verifier of round 1 noted; here the shanks outside that block are read, and the LFP notebook's
in-block result is combined with them in the record (the in-block shanks are not read again, to keep the run under an hour). The other readings of the claim, and
every other claim, are in the other notebooks (claims table in the main notebook's header).

## Setup
Every file of the pinned version is streamed; the spikes, units, event tables and electrode flags are read into memory.

In [ ]:
import zlib, time, resource, collections, warnings
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import tingley2019 as T
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
KEY = "tingley-2019"
def seed_of(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
t0 = time.time()
A = T.list_assets()
S = T.load_all(A, workers=8)
print(f"{len(S)} files of dandiset {T.DANDISET} version {T.VERSION} loaded in {time.time()-t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Data check
The recordings and LS neurons used, with the shanks and storage blocks to be read.

In [ ]:
def ls_ev(s):
    return s.events["LSRipples"] if "LSRipples" in s.events and len(s.events["LSRipples"]) else None
def det_row(s, name):
    r = np.flatnonzero(s.electrodes.ripple_detection_channel.str.contains(name).to_numpy())
    return int(r[0]) if len(r) else None
LFP_SESS = [s for s in S if det_row(s, "LSRipples") is not None and len(s.units_in(["ls"])) and ls_ev(s) is not None]
FS = T.FS_LFP
def shank_rows(s):
    """Per LS shank, the electrode row read for it: the middle site of the shank among those in the detection channel's block (the LFP
    notebook's rule); for a shank with no site in that block, the middle of its sites in channel order."""
    ls_units = s.units_in(["ls"]); sg = T.electrode_spike_groups(s.url); el, cw = T.lfp_columns(s.url)
    colof = {int(e): i for i, e in enumerate(el)}; det = det_row(s, "LSRipples"); blk = colof[det] // cw
    out = {}
    for sh in np.unique(s.unit_shank[ls_units]):
        cand = sorted((colof[r], r) for r in np.flatnonzero(sg == sh) if r in colof)
        inb = [r for c, r in cand if c // cw == blk]
        use = inb if inb else [r for c, r in cand]
        if use: out[int(sh)] = (int(use[len(use) // 2]), bool(inb), colof[use[len(use) // 2]] // cw)
    return out
SR = {s.session: shank_rows(s) for s in LFP_SESS}
nu = sum(len(s.units_in(["ls"])) for s in LFP_SESS)
cov = sum(int(np.isin(s.unit_shank[s.units_in(["ls"])], list(SR[s.session])).sum()) for s in LFP_SESS)
inb = sum(int(np.isin(s.unit_shank[s.units_in(["ls"])], [k for k, v in SR[s.session].items() if v[1]]).sum()) for s in LFP_SESS)
print(f"Recordings: {len(LFP_SESS)} ({collections.Counter(s.subject for s in LFP_SESS)}); LS neurons {nu}; on a shank with an LFP site {cov}; "
      f"on a shank with a site in the detection channel's block {inb}")
print(f"Blocks to read (shanks outside the detection block): {sum(len({v[2] for v in SR[s.session].values() if not v[1]}) for s in LFP_SESS)}; LFP block-hours "
      f"{sum(len({v[2] for v in SR[s.session].values() if not v[1]}) * s.duration for s in LFP_SESS) / 3600:.0f}")
print("Region label of the sites read:", collections.Counter(s.electrodes.region[v[0]] for s in LFP_SESS for v in SR[s.session].values()))

## fig1d-phase-locking, own-shank reading for every LS neuron
> The vast majority of LS neurons (1840 of 1989, 92%) are significantly phase-locked to the 120-200 Hz band (Rayleigh P < 0.05),
> with a strong unimodal preference for the trough of each cycle.

As in the LFP notebook (STAR Methods, Phase locking analysis): epochs where the 120-200 Hz power (4th-order Butterworth, 7 ms RMS) is
continuously above 2 SD; the Hilbert phase of every spike in them; neurons with more than 50 spikes; Rayleigh p < 0.05; a null of 10
circular shifts (a second generator of the claim, its seed plus 1, and the recording's index). Each storage block is read once, keeping
only the sites used. Only shanks with no site in the detection channel's block are read here.

In [ ]:
for s in S:
    for u in np.flatnonzero(s.unit_region != "ls"): s.spikes[u] = np.zeros(0)
SEED = seed_of("fig1d-phase-locking") + 1
def job(k):
    s = LFP_SESS[k]; t0 = time.time(); rng = np.random.default_rng([SEED, k])
    ls_units = s.units_in(["ls"]); rows = []
    by_blk = collections.defaultdict(list)
    for sh, (r, inb, b) in SR[s.session].items():
        if not inb: by_blk[b].append((sh, r, inb))     # shanks in the detection block are in the LFP notebook
    for b, lst in sorted(by_blk.items()):
        X, kept, _ = T.read_lfp_block(s.url, lst[0][1], [r for _, r, _ in lst]); kept = list(kept)
        for sh, r, inb in lst:
            fl = T.bandpass(X[:, kept.index(r)].astype(np.float32), 120, 200, 4); ph = np.angle(T.analytic(fl)).astype(np.float32); m = T.high_power_epochs(fl)
            sel = np.flatnonzero(s.unit_shank[ls_units] == sh)
            for j, u in zip(sel, T.lock_units([s.spikes[ls_units[j]] for j in sel], ph, m, rng)):
                rows.append(dict(session=s.session, subject=s.subject, unit=int(ls_units[j]), shank=sh, in_det_block=inb, **u))
            del fl, ph, m
        del X
    return rows, time.time() - t0
# Processes, not threads: h5py serializes reads within one process. Forked workers share the loaded spikes.
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
t0 = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    RES = list(ex.map(job, range(len(LFP_SESS))))
OWN = pd.DataFrame([r for rows, _ in RES for r in rows])
print(f"{len(LFP_SESS)} recordings in {(time.time()-t0)/60:.1f} min (median {np.median([x for _, x in RES]):.0f} s); "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB (largest worker {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1e6:.2f} GB)")
for nm, D in [("shanks outside the detection channel's block", OWN)]:
    sel = D[D.n > 50]; sig = sel[sel.p < 0.05]
    null = np.nanmean(np.vstack(sel.p_null.values) < 0.05) if len(sel) else np.nan
    mang = np.angle(np.exp(1j * sig.angle).mean()); rt = T.rayleigh(sig.angle.to_numpy())
    print(f"{nm}: LS neurons {len(D)}; with > 50 spikes {len(sel)}; significant {len(sig)}; fraction (a) {len(sig)/max(len(sel),1):.3f}, (b) {len(sig)/max(len(D),1):.3f}; "
          f"null fraction {null:.3f}; preferred phase {np.degrees(mang):.1f} deg (|distance to 180| {180-abs(np.degrees(mang)):.1f}), Rayleigh over neurons p = {rt[0]:.2g}")
sel = OWN[OWN.n > 50]
print("Per animal, shanks outside the block:", sel.assign(sig=sel.p < .05).groupby("subject").sig.agg(["count", "mean"]).round(3).to_dict("index"))
print(f"Resultant length range {sel.r.min():.3f} to {sel.r.max():.3f}")

**Result: fig1d-phase-locking, own-shank reading for the shanks outside the detection channel's block.** Of 1028 LS neurons on these shanks, 863 have more than 50 spikes in high-power epochs and 732 of them (84.8%) are significantly locked; 10 circular shifts give about 5%; significant neurons prefer -167.7 degrees. With the 883 of 1129 neurons of the in-block shanks (LFP notebook), the own-shank reading covers 1992 neurons of which 1615 (81.1%) are locked, below the tolerance's lower bound of 83.25%. The outcome is in the LFP notebook.

## Summary

The outcomes of all claims, with how the analysis differs from the paper's and what was found about the dataset, are in the summary of `reproduce_tingley_2019.ipynb`.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.